# 14. Staged fine-tuning V3

Этот ноутбук сохраняет проверенную реализацию исходного эксперимента и имена его
артефактов. Запускайте **Restart Kernel and Run All Cells** после выполнения всех
предыдущих пронумерованных ноутбуков.

Все входы, кроме исходных raw-данных из `config/raw_sources.json`, создаются внутри
этого проекта. Результаты записываются в `outputs/`, а модели — в `checkpoints/`.

In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Неверный путь.")

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from project_paths import *
ensure_project_directories()

print("Project root:", PROJECT_ROOT)

Project root: D:\Users\user\Desktop\DS_XRD_project


# Fine-tuning v3: staged real adaptation

Загрузка готового synthetic pretrain v2 → 30 эпох FT с replay 50/50 → 20 эпох только на real с пониженным LR. Сравнение проводится на тех же 5 GroupKFold-фолдах, что и v2.

In [2]:
import json
import math
import random
import time
from collections import defaultdict
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE, '|', torch.cuda.get_device_name(0))

device: cuda | NVIDIA GeForce RTX 3060 Ti


In [3]:
MODE = 'cv'   # 'cv' — 5 фолдов + финальная модель; 'quick' — 1 фолд для проверки

BASE = PROJECT_ROOT
DATA = BASE / 'data' / 'preprocessed'
CKPT_PRE = BASE / 'checkpoints' / 'pretrain_v2_full_best.pt'
OUT = BASE / 'outputs'
CKPT_DIR = BASE / 'checkpoints'

GRID_N = 4096
W = 3
SYSTEMS = ['triclinic', 'monoclinic', 'orthorhombic', 'tetragonal',
           'trigonal', 'hexagonal', 'cubic']
IMPUTE_LAMBDA = 1.5406
W_LAT, W_VOL, W_SG, W_SYS, W_EL, W_ANG = 1.0, 0.5, 1.0, 1.0, 1.0, 2.0

# Общий бюджет ровно как у FT v2: 50 эпох.
# Стадия 2 намеренно понижает LR в 3 раза: это адаптация к real, а не переобучение с нуля.
HP = dict(
    batch=128,
    lr_head_stage1=3e-5,
    lr_backbone_stage1=1e-5,
    lr_head_stage2=1e-5,
    lr_backbone_stage2=3e-6,
    wd=1e-4,
    clip=1.0,
    epochs_stage1=30,
    epochs_stage2=20,
    replay_frac=0.5,
    warmup_frac=0.1,
    n_folds=5,
)
assert HP['epochs_stage1'] + HP['epochs_stage2'] == 50

In [4]:
stats = json.loads((OUT / 'pretrain_stats.json').read_text())
VOCAB = stats['vocab']
EL_IDX = {e: i for i, e in enumerate(VOCAB)}
LAT_MEAN = np.array(stats['lat_mean'])
LAT_STD = np.array(stats['lat_std'])
VOL_MEAN, VOL_STD = stats['vol_mean'], stats['vol_std']

index = pd.read_parquet(DATA / 'index_preprocessed.parquet')
N_TOTAL = len(index)
X_MM = np.memmap(DATA / 'X_intensity.f16', dtype=np.float16, mode='r', shape=(N_TOTAL, GRID_N))
M_MM = np.memmap(DATA / 'M_mask.u8', dtype=np.uint8, mode='r', shape=(N_TOTAL, GRID_N))
row_of = dict(zip(index['sample_id'], index['row_idx']))

ft = pd.read_parquet(BASE / 'data' / 'clean' / 'ft_pool_combined.parquet')
ft['row_idx'] = ft['sample_id'].map(row_of)
assert ft['row_idx'].notna().all()
ft['primary_wavelength'] = ft['primary_wavelength'].fillna(IMPUTE_LAMBDA)

abc = ft[['lattice_a', 'lattice_b', 'lattice_c']].to_numpy(np.float64)
ang = ft[['alpha', 'beta', 'gamma']].to_numpy(np.float64)
ca, cb, cg = (np.cos(np.radians(ang[:, i])) for i in range(3))
t = 1.0 - ca**2 - cb**2 - cg**2 + 2 * ca * cb * cg
ft['V'] = abc.prod(1) * np.sqrt(np.clip(t, 1e-12, None))
ft['lat6'] = list(np.hstack([np.log(abc), ang]))

def to_list(v):
    if isinstance(v, str):
        try:
            return json.loads(v)
        except Exception:
            return []
    if isinstance(v, (list, tuple, np.ndarray)):
        return [e for e in v if isinstance(e, str)]
    return []

ft['elements'] = ft['elements_list'].apply(to_list)
ft['conn_key'] = (ft['phase_compositions'].astype(str) + '|'
                  + ft['lattice_a'].round(2).astype(str))
print('FT-пул:', len(ft), '| уникальных соединений:', ft['conn_key'].nunique())

# replay-пул (трейн-часть синтетики, как в v1)
splits_pre = pd.read_parquet(OUT / 'splits_pretrain.parquet')
syn_meta = pd.read_parquet(BASE / 'data' / 'clean' / 'df_synth_summary_final_clean.parquet')[
    ['sample_id', 'lattice_a', 'lattice_b', 'lattice_c',
     'alpha', 'beta', 'gamma', 'spacegroup_number', 'crystal_system', 'elements_list']
]
pre_rows = index[index['split_role'] == 'pretrain'][['sample_id', 'row_idx',
                                                      'lambda_1', 'lambda_2']]
syn = pre_rows.merge(splits_pre, on='sample_id').merge(syn_meta, on='sample_id')
syn = syn[syn['split'] == 'train'].reset_index(drop=True)
abc_s = syn[['lattice_a', 'lattice_b', 'lattice_c']].to_numpy(np.float64)
ang_s = syn[['alpha', 'beta', 'gamma']].to_numpy(np.float64)
ca, cb, cg = (np.cos(np.radians(ang_s[:, i])) for i in range(3))
t = 1.0 - ca**2 - cb**2 - cg**2 + 2 * ca * cb * cg
syn['V'] = abc_s.prod(1) * np.sqrt(np.clip(t, 1e-12, None))
syn['lat6'] = list(np.hstack([np.log(abc_s), ang_s]))
syn['elements'] = syn['elements_list'].apply(to_list)
print('replay-пул:', len(syn))

FT-пул: 3475 | уникальных соединений: 2029
replay-пул: 451027


In [5]:
# ---------- датасеты и лоадеры ----------
def build_labels(frame, is_real):
    lam1_col = 'primary_wavelength' if 'primary_wavelength' in frame else 'lambda_1'
    lam2_col = 'secondary_wavelength' if 'secondary_wavelength' in frame else 'lambda_2'
    lam1 = frame[lam1_col].to_numpy(np.float32)
    lam2 = frame[lam2_col].to_numpy(np.float32)
    lam = np.stack([lam1 / 1.54, np.nan_to_num(lam2) / 1.54,
                    np.isfinite(lam2).astype(np.float32)], 1)
    lat6 = np.stack(frame['lat6'].to_numpy())
    has_lat = ~np.isnan(lat6).any(1)
    latm = has_lat.astype(np.float32)
    lat6 = (lat6 - LAT_MEAN) / LAT_STD
    vol = (np.log(frame['V'].to_numpy(np.float64)) - VOL_MEAN) / VOL_STD
    sg_raw = frame['spacegroup_number'].to_numpy(float)
    sgm = np.isfinite(sg_raw).astype(np.float32)
    sg = np.nan_to_num(sg_raw).astype(np.int64) - 1
    sysmap = {s: i for i, s in enumerate(SYSTEMS)}
    sys_raw = frame['crystal_system'].map(sysmap)
    sysm = sys_raw.notna().to_numpy(np.float32)
    sys_ = sys_raw.fillna(0).to_numpy(np.int64)
    n = len(frame)
    el = np.zeros((n, len(VOCAB)), np.float32)
    elm = np.zeros(n, np.float32)
    for i, els in enumerate(frame['elements']):
        if len(els):
            elm[i] = 1.0
            for e in els:
                j = EL_IDX.get(e)
                if j is not None:
                    el[i, j] = 1.0
    return dict(lam=lam, lat6=lat6.astype(np.float32), latm=latm,
                vol=vol.astype(np.float32), volm=latm.copy(),
                sg=sg, sgm=sgm, sys_=sys_, sysm=sysm, el=el, elm=elm)

class SpecDS(Dataset):
    def __init__(self, frame, is_real):
        self.row = frame['row_idx'].to_numpy(np.int64)
        self.L = build_labels(frame, is_real)

    def __len__(self):
        return len(self.row)

    def __getitem__(self, i):
        x = np.empty((2, GRID_N), np.float32)
        x[0] = X_MM[self.row[i]]
        x[1] = M_MM[self.row[i]]
        L = self.L
        return (torch.from_numpy(x), torch.from_numpy(L['lam'][i]),
                torch.from_numpy(L['lat6'][i]), torch.tensor(L['latm'][i]),
                torch.tensor(L['sg'][i]), torch.tensor(L['sgm'][i]),
                torch.tensor(L['sys_'][i]), torch.tensor(L['sysm'][i]),
                torch.from_numpy(L['el'][i]), torch.tensor(L['elm'][i]),
                torch.tensor(L['vol'][i]), torch.tensor(L['volm'][i]))

class MixedLoader:
    def __init__(self, real_ds, syn_ds, batch, frac):
        self.real, self.syn, self.batch, self.frac = real_ds, syn_ds, batch, frac
        self.n_syn = max(1, int(batch * frac))
        self.n_real = max(1, batch - self.n_syn)
        self.steps = max(1, len(real_ds) // self.n_real)

    def __len__(self):
        return self.steps

    def __iter__(self):
        real_idx = np.random.permutation(len(self.real))
        for b in range(self.steps):
            r = real_idx[b * self.n_real:(b + 1) * self.n_real]
            s = np.random.randint(0, len(self.syn), size=self.n_syn)
            items = [self.real[i] for i in r] + [self.syn[j] for j in s]
            yield [torch.stack(t) for t in zip(*items)]

In [6]:
# ---------- модель v2 ----------
class ResBlock(nn.Module):
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(cin, cout, 3, stride=stride, padding=1, bias=False)
        self.n1 = nn.GroupNorm(8, cout)
        self.conv2 = nn.Conv1d(cout, cout, 3, padding=1, bias=False)
        self.n2 = nn.GroupNorm(8, cout)
        if cin == cout and stride == 1:
            self.skip = nn.Identity()
        else:
            self.skip = nn.Sequential(nn.Conv1d(cin, cout, 1, stride=stride, bias=False),
                                      nn.GroupNorm(8, cout))

    def forward(self, x):
        h = F.gelu(self.n1(self.conv1(x)))
        h = self.n2(self.conv2(h))
        return F.gelu(h + self.skip(x))

class XRDNetV2(nn.Module):
    def __init__(self, n_el, w=3):
        super().__init__()
        c = [32 * w, 48 * w, 64 * w, 96 * w, 128 * w, 192 * w, 256 * w]
        self.stem = nn.Sequential(nn.Conv1d(2, c[0], 15, padding=7, bias=False),
                                  nn.GroupNorm(8, c[0]), nn.GELU())
        self.blocks = nn.Sequential(*[ResBlock(c[i], c[i + 1], stride=2)
                                      for i in range(6)])
        self.lam_mlp = nn.Sequential(nn.Linear(3, 16 * w), nn.GELU(), nn.Linear(16 * w, 16 * w))
        self.trunk = nn.Sequential(nn.Linear(c[-1] + 16 * w, 512 * w), nn.GELU(),
                                   nn.Linear(512 * w, 512 * w), nn.GELU())
        self.head_lat = nn.Linear(512 * w, 6)
        self.head_vol = nn.Linear(512 * w, 1)
        self.head_sg = nn.Linear(512 * w, 230)
        self.head_sys = nn.Linear(512 * w, 7)
        self.head_el = nn.Linear(512 * w, n_el)

    def forward(self, x, lam):
        f = self.stem(x)
        f = self.blocks(f)
        w = F.adaptive_avg_pool1d(x[:, 1:2], f.shape[-1]).clamp_min(1e-3)
        pooled = (f * w).sum(-1) / w.sum(-1)
        z = torch.cat([pooled, self.lam_mlp(lam)], dim=1)
        z = self.trunk(z)
        return dict(lat=self.head_lat(z), vol=self.head_vol(z).squeeze(-1),
                    sg=self.head_sg(z), sys=self.head_sys(z), el=self.head_el(z))

def load_pretrained():
    m = XRDNetV2(len(VOCAB), w=W).to(DEVICE)
    sd = torch.load(CKPT_PRE, map_location=DEVICE, weights_only=True)
    m.load_state_dict(sd)
    return m

print('модель v2 готова к загрузке чекпойнта')

модель v2 готова к загрузке чекпойнта


In [7]:
# ---------- лоссы, метрики ----------
def masked_l1(pred, tgt, mask):
    m = mask > 0
    if m.sum() == 0:
        return pred.new_zeros(())
    return F.smooth_l1_loss(pred[m], tgt[m])

def masked_ce(logits, tgt, mask):
    m = mask > 0
    if m.sum() == 0:
        return logits.new_zeros(())
    return F.cross_entropy(logits[m], tgt[m].long())

def compute_losses(out, batch):
    (_, _, lat, latm, sg, sgm, sys_, sysm, el, elm, vol, volm) = batch
    m = latm > 0
    if m.sum() > 0:
        l_len = F.smooth_l1_loss(out['lat'][m][:, :3], lat[m][:, :3])
        l_ang = F.smooth_l1_loss(out['lat'][m][:, 3:], lat[m][:, 3:])
    else:
        l_len = l_ang = out['lat'].new_zeros(())
    if elm.sum() > 0:
        me = (elm > 0)
        loss_el = F.binary_cross_entropy_with_logits(out['el'][me], el[me])
    else:
        loss_el = out['el'].new_zeros(())
    losses = dict(lat=l_len, ang=l_ang,
                  vol=masked_l1(out['vol'], vol, volm),
                  sg=masked_ce(out['sg'], sg, sgm),
                  sys=masked_ce(out['sys'], sys_, sysm),
                  el=loss_el)
    weighted = (W_LAT * l_len + W_ANG * l_ang + W_VOL * losses['vol']
                + W_SG * losses['sg'] + W_SYS * losses['sys'] + W_EL * loss_el)
    return losses, weighted

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    rows = []
    for batch in loader:
        batch = [b.to(DEVICE, non_blocking=True) for b in batch]
        with torch.autocast('cuda', dtype=torch.float16):
            out = model(batch[0], batch[1])
        _, _, lat, latm, sg, sgm, sys_, sysm, el, elm, vol, volm = batch
        lat_pred = out['lat'].float().cpu().numpy() * LAT_STD + LAT_MEAN
        lat_pred[:, :3] = np.exp(lat_pred[:, :3])
        lat_true = lat.cpu().numpy() * LAT_STD + LAT_MEAN
        lat_true[:, :3] = np.exp(lat_true[:, :3])
        for i in range(len(lat)):
            rows.append(dict(
                latm=float(latm[i]), sgm=float(sgm[i]), sysm=float(sysm[i]),
                elm=float(elm[i]),
                sg_ok=float(sgm[i] > 0 and out['sg'][i].argmax().item() == sg[i].item()),
                sg_top5=float(sgm[i] > 0 and sg[i].item() in out['sg'][i].topk(5).indices.tolist()),
                sys_ok=float(sysm[i] > 0 and out['sys'][i].argmax().item() == sys_[i].item()),
                mae_a=abs(lat_pred[i, 0] - lat_true[i, 0]) if latm[i] > 0 else np.nan,
                mae_b=abs(lat_pred[i, 1] - lat_true[i, 1]) if latm[i] > 0 else np.nan,
                mae_c=abs(lat_pred[i, 2] - lat_true[i, 2]) if latm[i] > 0 else np.nan,
                mae_ang=float(np.abs(lat_pred[i, 3:] - lat_true[i, 3:]).mean()) if latm[i] > 0 else np.nan,
                pred_el=frozenset(np.where(torch.sigmoid(out['el'][i]).cpu().numpy() > 0.5)[0]),
                true_el=frozenset(np.where(el[i].cpu().numpy() > 0.5)[0]) if elm[i] > 0 else frozenset(),
                elm_f=float(elm[i]),
            ))
    model.train()
    d = pd.DataFrame(rows)
    res = {}
    v = d[d['sgm'] > 0]
    res['sg_acc'] = v['sg_ok'].mean() if len(v) else np.nan
    res['sg_top5'] = v['sg_top5'].mean() if len(v) else np.nan
    res['sg_n'] = len(v)
    v = d[d['sysm'] > 0]
    res['sys_acc'] = v['sys_ok'].mean() if len(v) else np.nan
    v = d[d['latm'] > 0]
    if len(v):
        res['mae_a'] = v['mae_a'].mean()
        res['mae_a_med'] = v['mae_a'].median()
        res['mae_ang'] = v['mae_ang'].mean()
    v = d[(d['elm'] > 0)]
    if len(v):
        tp = sum(len(r['pred_el'] & r['true_el']) for _, r in v.iterrows())
        fp = sum(len(r['pred_el'] - r['true_el']) for _, r in v.iterrows())
        fn = sum(len(r['true_el'] - r['pred_el']) for _, r in v.iterrows())
        p = tp / max(tp + fp, 1)
        rc = tp / max(tp + fn, 1)
        res['el_f1_micro'] = 2 * p * rc / max(p + rc, 1e-9)
        res['el_exact'] = float((v['pred_el'] == v['true_el']).mean())
    res['n'] = len(d)
    return res, d

In [8]:
# ---------- staged fine-tuning одного фолда ----------
def make_optimizer(model, lr_backbone, lr_head):
    backbone = [p for n, p in model.named_parameters()
                if not n.startswith(('head_', 'trunk', 'lam_mlp'))]
    heads = [p for n, p in model.named_parameters()
             if n.startswith(('head_', 'trunk', 'lam_mlp'))]
    return torch.optim.AdamW([
        {'params': backbone, 'lr': lr_backbone},
        {'params': heads, 'lr': lr_head},
    ], weight_decay=HP['wd'])

def run_stage(model, loader, val_loader, epochs, lr_backbone, lr_head,
              stage_name, best_score, best_epoch, best_state, epoch_offset, verbose=True):
    opt = make_optimizer(model, lr_backbone, lr_head)
    scaler = torch.amp.GradScaler('cuda')
    steps_total = epochs * len(loader)
    warmup = max(1, int(steps_total * HP['warmup_frac']))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (
        s / warmup if s < warmup else
        0.5 * (1 + math.cos(math.pi * min((s - warmup) / max(steps_total - warmup, 1), 1)))))

    for epoch in range(epochs):
        model.train()
        for batch in loader:
            batch = [b.to(DEVICE, non_blocking=True) for b in batch]
            with torch.autocast('cuda', dtype=torch.float16):
                out = model(batch[0], batch[1])
                _, total = compute_losses(out, batch)
            opt.zero_grad(set_to_none=True)
            scaler.scale(total).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), HP['clip'])
            scaler.step(opt)
            scaler.update()
            sched.step()

        m, _ = evaluate(model, val_loader)
        score = ((m.get('sys_acc') or 0) + (m.get('el_f1_micro') or 0)
                 + 0.5 * (m.get('sg_acc') or 0))
        global_epoch = epoch_offset + epoch + 1
        if verbose:
            print(f"  {stage_name} {epoch+1:2d}/{epochs} (global {global_epoch:2d}) | "
                  f"sys {m.get('sys_acc', float('nan')):.3f} "
                  f"sg {m.get('sg_acc', float('nan')):.3f} "
                  f"el {m.get('el_f1_micro', float('nan')):.3f} "
                  f"mae_a {m.get('mae_a', float('nan')):.2f}", flush=True)
        if score > best_score:
            best_score, best_epoch = score, global_epoch
            best_state = {k: v.detach().cpu().clone()
                          for k, v in model.state_dict().items()}
    return best_score, best_epoch, best_state

def train_fold_staged(train_frame, val_frame, verbose=True):
    """pretrained -> replay 50/50 -> real-only low-LR; best checkpoint выбирается по real-val."""
    model = load_pretrained()
    real_ds = SpecDS(train_frame, True)
    val_loader = DataLoader(SpecDS(val_frame, True), batch_size=HP['batch'],
                            shuffle=False, pin_memory=True)
    best_score, best_epoch, best_state = -1.0, 0, None

    # Стадия 1: сохранение synthetic-признаков при адаптации к реальному домену.
    replay_loader = MixedLoader(real_ds, SpecDS(syn, False), HP['batch'], HP['replay_frac'])
    best_score, best_epoch, best_state = run_stage(
        model, replay_loader, val_loader, HP['epochs_stage1'],
        HP['lr_backbone_stage1'], HP['lr_head_stage1'], 'replay',
        best_score, best_epoch, best_state, 0, verbose)

    # Стадия 2: никакой синтетики; осторожная адаптация к реальным спектрам.
    real_loader = DataLoader(real_ds, batch_size=HP['batch'], shuffle=True, pin_memory=True)
    best_score, best_epoch, best_state = run_stage(
        model, real_loader, val_loader, HP['epochs_stage2'],
        HP['lr_backbone_stage2'], HP['lr_head_stage2'], 'real-only',
        best_score, best_epoch, best_state, HP['epochs_stage1'], verbose)

    model.load_state_dict(best_state)
    return model, best_epoch, best_score

def train_final_staged(all_frame):
    """Финальная модель: фиксированные 30+20 эпох, без использования validation-меток."""
    model = load_pretrained()
    real_ds = SpecDS(all_frame, True)

    def train_only(loader, epochs, lr_backbone, lr_head, name):
        opt = make_optimizer(model, lr_backbone, lr_head)
        scaler = torch.amp.GradScaler('cuda')
        total = epochs * len(loader)
        warmup = max(1, int(total * HP['warmup_frac']))
        sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (
            s / warmup if s < warmup else
            0.5 * (1 + math.cos(math.pi * min((s-warmup)/max(total-warmup, 1), 1)))))
        for epoch in range(epochs):
            model.train()
            for batch in loader:
                batch = [b.to(DEVICE, non_blocking=True) for b in batch]
                with torch.autocast('cuda', dtype=torch.float16):
                    _, loss = compute_losses(model(batch[0], batch[1]), batch)
                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), HP['clip'])
                scaler.step(opt)
                scaler.update()
                sched.step()
            print(f'final {name}: {epoch+1}/{epochs}', flush=True)

    train_only(MixedLoader(real_ds, SpecDS(syn, False), HP['batch'], HP['replay_frac']),
               HP['epochs_stage1'], HP['lr_backbone_stage1'], HP['lr_head_stage1'], 'replay')
    train_only(DataLoader(real_ds, batch_size=HP['batch'], shuffle=True, pin_memory=True),
               HP['epochs_stage2'], HP['lr_backbone_stage2'], HP['lr_head_stage2'], 'real-only')
    return model

In [9]:
# ---------- 5-fold GroupKFold: staged FT ----------
from sklearn.model_selection import GroupKFold

gkf = GroupKFold(n_splits=HP['n_folds'])
folds = list(gkf.split(ft, groups=ft['conn_key']))
print('фолды идентичны v2: группировка по conn_key, утечки исключены конструктивно')
for k, (tr_i, va_i) in enumerate(folds):
    print(f'  fold {k+1}: train {len(tr_i)} / val {len(va_i)}')

fold_metrics, best_epochs, zero_shot_metrics = [], [], []
t0 = time.time()
folds_to_run = folds[:1] if MODE == 'quick' else folds
for k, (tr_i, va_i) in enumerate(folds_to_run):
    print(f'\n===== STAGED FOLD {k+1}/{len(folds_to_run)} =====')
    train_frame = ft.iloc[tr_i].reset_index(drop=True)
    val_frame = ft.iloc[va_i].reset_index(drop=True)

    zs_model = load_pretrained()
    zs_m, _ = evaluate(zs_model, DataLoader(SpecDS(val_frame, True), batch_size=HP['batch'],
                                              shuffle=False, pin_memory=True))
    zero_shot_metrics.append(zs_m)
    print(f"  zero-shot: sys {zs_m.get('sys_acc', float('nan')):.3f} "
          f"el {zs_m.get('el_f1_micro', float('nan')):.3f} "
          f"mae_a {zs_m.get('mae_a', float('nan')):.2f}")
    del zs_model
    torch.cuda.empty_cache()

    model, best_ep, _ = train_fold_staged(train_frame, val_frame)
    best_epochs.append(best_ep)
    m, _ = evaluate(model, DataLoader(SpecDS(val_frame, True), batch_size=HP['batch'],
                                      shuffle=False, pin_memory=True))
    fold_metrics.append(m)
    print(f"  best global epoch {best_ep} | итог: sys {m.get('sys_acc', float('nan')):.3f} "
          f"sg {m.get('sg_acc', float('nan')):.3f} el {m.get('el_f1_micro', float('nan')):.3f} "
          f"mae_a {m.get('mae_a', float('nan')):.2f}")
    del model
    torch.cuda.empty_cache()

print(f'\nCV заняла {(time.time()-t0)/60:.1f} мин')

фолды идентичны v2: группировка по conn_key, утечки исключены конструктивно
  fold 1: train 2780 / val 695
  fold 2: train 2780 / val 695
  fold 3: train 2780 / val 695
  fold 4: train 2780 / val 695
  fold 5: train 2780 / val 695

===== STAGED FOLD 1/5 =====
  zero-shot: sys 0.323 el 0.166 mae_a 3.41
  replay  1/30 (global  1) | sys 0.388 sg 0.059 el 0.168 mae_a 3.37
  replay  2/30 (global  2) | sys 0.442 sg 0.126 el 0.177 mae_a 3.30
  replay  3/30 (global  3) | sys 0.490 sg 0.178 el 0.201 mae_a 3.23
  replay  4/30 (global  4) | sys 0.504 sg 0.207 el 0.232 mae_a 3.18
  replay  5/30 (global  5) | sys 0.529 sg 0.252 el 0.254 mae_a 3.12
  replay  6/30 (global  6) | sys 0.538 sg 0.267 el 0.279 mae_a 3.07
  replay  7/30 (global  7) | sys 0.562 sg 0.267 el 0.304 mae_a 3.04
  replay  8/30 (global  8) | sys 0.562 sg 0.289 el 0.317 mae_a 3.00
  replay  9/30 (global  9) | sys 0.575 sg 0.289 el 0.332 mae_a 2.96
  replay 10/30 (global 10) | sys 0.579 sg 0.296 el 0.341 mae_a 2.94
  replay 11/30 (g

In [10]:
# ---------- сводка и сравнение с v2 ----------
keys = ['sys_acc', 'sg_acc', 'sg_top5', 'el_f1_micro', 'el_exact',
        'mae_a', 'mae_a_med', 'mae_ang']
rows = []
for k in keys:
    zs = [m.get(k) for m in zero_shot_metrics]
    staged = [m.get(k) for m in fold_metrics]
    zs_v = [v for v in zs if v == v]
    st_v = [v for v in staged if v == v]
    rows.append(dict(
        metric=k,
        zero_shot=f'{np.mean(zs_v):.3f}' if zs_v else '-',
        staged_ft=f'{np.mean(st_v):.3f}' if st_v else '-',
        std=f'{np.std(st_v):.3f}' if len(st_v) > 1 else '-',
        folds=len(st_v),
    ))
summary = pd.DataFrame(rows)
print(summary.to_string(index=False))
if len(fold_metrics) > 1:
    print('\nлучшие global-эпохи:', best_epochs)
summary.to_csv(OUT / 'ft_v3_staged_cv_summary.csv', index=False)

v2_path = OUT / 'ft_v2_cv_summary.csv'
if v2_path.exists():
    v2 = pd.read_csv(v2_path)[['metric', 'after_ft']]
    comparison = summary[['metric', 'staged_ft']].merge(v2, on='metric', how='outer')
    comparison['staged_ft'] = pd.to_numeric(comparison['staged_ft'], errors='coerce')
    comparison['v2_ft'] = pd.to_numeric(comparison['after_ft'], errors='coerce')
    comparison['delta_v3_minus_v2'] = comparison['staged_ft'] - comparison['v2_ft']
    print('\n===== V3 STAGED vs V2 50/50 =====')
    print(comparison[['metric', 'staged_ft', 'v2_ft', 'delta_v3_minus_v2']].to_string(index=False))
    comparison.to_csv(OUT / 'ft_v3_staged_vs_v2.csv', index=False)

     metric zero_shot staged_ft   std  folds
    sys_acc     0.345     0.628 0.019      5
     sg_acc     0.069     0.510 0.221      5
    sg_top5     0.143     0.704 0.229      5
el_f1_micro     0.181     0.490 0.040      5
   el_exact     0.002     0.167 0.041      5
      mae_a     3.466     2.707 0.179      5
  mae_a_med     2.178     1.559 0.121      5
    mae_ang     6.034     4.661 0.117      5

лучшие global-эпохи: [49, 45, 43, 47, 40]

===== V3 STAGED vs V2 50/50 =====
     metric  staged_ft  v2_ft  delta_v3_minus_v2
   el_exact      0.167  0.179             -0.012
el_f1_micro      0.490  0.501             -0.011
      mae_a      2.707  2.662              0.045
  mae_a_med      1.559  1.523              0.036
    mae_ang      4.661  4.574              0.087
     sg_acc      0.510  0.509              0.001
    sg_top5      0.704  0.676              0.028
    sys_acc      0.628  0.630             -0.002


In [11]:
# ---------- финальная staged-модель на всём реальном пуле ----------
if MODE == 'cv':
    print(f"финальное staged-обучение: весь пул {len(ft)} | "
          f"{HP['epochs_stage1']} эпох replay + {HP['epochs_stage2']} эпох real-only")
    model = train_final_staged(ft.reset_index(drop=True))
    torch.save(model.state_dict(), CKPT_DIR / 'ft_v3_staged_final.pt')
    print('сохранено: checkpoints/ft_v3_staged_final.pt')
else:
    print('MODE=quick: финальная модель не обучается')

финальное staged-обучение: весь пул 3475 | 30 эпох replay + 20 эпох real-only
final replay: 1/30
final replay: 2/30
final replay: 3/30
final replay: 4/30
final replay: 5/30
final replay: 6/30
final replay: 7/30
final replay: 8/30
final replay: 9/30
final replay: 10/30
final replay: 11/30
final replay: 12/30
final replay: 13/30
final replay: 14/30
final replay: 15/30
final replay: 16/30
final replay: 17/30
final replay: 18/30
final replay: 19/30
final replay: 20/30
final replay: 21/30
final replay: 22/30
final replay: 23/30
final replay: 24/30
final replay: 25/30
final replay: 26/30
final replay: 27/30
final replay: 28/30
final replay: 29/30
final replay: 30/30
final real-only: 1/20
final real-only: 2/20
final real-only: 3/20
final real-only: 4/20
final real-only: 5/20
final real-only: 6/20
final real-only: 7/20
final real-only: 8/20
final real-only: 9/20
final real-only: 10/20
final real-only: 11/20
final real-only: 12/20
final real-only: 13/20
final real-only: 14/20
final real-only: 1

## Интерпретация

Сравнение с `ft_v2_cv_summary.csv` валидно, поскольку используются тот же pretrain checkpoint, те же реальные фолды, архитектура, маски и метрики. Меняется только schedule FT: 30 эпох replay 50/50, затем 20 эпох real-only с LR / 3.